# StartupFund AI - Complete Guide

This notebook provides a comprehensive guide to the StartupFund AI ML pipeline, including model training, evaluation, and deployment.

## 📋 Table of Contents

1. [Project Overview](#project-overview)
2. [Data Pipeline](#data-pipeline)
3. [Feature Engineering](#feature-engineering)
4. [Model Training](#model-training)
5. [Model Evaluation](#model-evaluation)
6. [Deployment](#deployment)
7. [Usage Examples](#usage-examples)

## Project Overview

**StartupFund AI** is a machine learning system designed to predict funding amounts for Indian startup investment rounds. The system uses historical data from 5,000+ investor records to estimate expected check sizes in Indian Rupees (₹).

### Key Features:
- **Multi-Model Comparison**: Ridge, Random Forest, and XGBoost models
- **Explainable AI**: SHAP-based feature attribution
- **Currency Support**: INR (Crore/Lakh formatting)
- **Production MLOps**: MLflow tracking, DVC version control, FastAPI API
- **Interactive UI**: Streamlit dashboard for predictions and analysis

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import sys

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully")

## Data Pipeline

### Data Ingestion and Validation

The data pipeline starts with loading the raw investor dataset and performing schema validation.

In [ ]:
from src.data.ingest_validate import load_raw_data, validate_schema, produce_data_summary

# Load raw data
raw_df = load_raw_data("Indian_Investor_Dataset_2026.csv")
print(f"📊 Loaded {len(raw_df)} investor records")

# Validate schema
is_valid, msg = validate_schema(raw_df)
print(f"✅ Schema Validation: {is_valid} - {msg}")

# Get data summary
summary = produce_data_summary(raw_df)
print(f"\n📈 Data Summary:")
print(f"- Columns: {summary['num_columns']}")
print(f"- Missing Values: {sum(summary['missing_values'].values())}")
print(f"- Duplicate Rows: {summary['duplicate_rows']}")

### Data Cleaning

The cleaning process includes:
- City name canonicalization
- Currency conversion (USD → INR)
- Binary flag conversion
- Data anomaly resolution

In [ ]:
from src.data.clean_data import clean_dataset

# Clean the dataset
cleaned_df = clean_dataset(raw_df)
print(f"✅ Data cleaned successfully")
print(f"📊 Cleaned dataset shape: {cleaned_df.shape}")

# Show sample of cleaned data
print("\n🔍 Sample cleaned data:")
print(cleaned_df.head(3))

## Feature Engineering

The system uses advanced feature engineering to create meaningful predictors:

### Engineered Features:
- **investor_age_years**: Years since investor was founded
- **exit_success_rate**: Ratio of successful exits to portfolio size
- **focus_sector_count**: Number of sector focus areas
- **investment_stage_rank**: Numerical representation of funding stage
- **portfolio_density**: Companies per year of operation
- **exit_intensity**: Exits per year of operation
- **Stage flags**: Early-stage vs late-stage investor indicators

In [ ]:
from src.features.build_features import StartupInvestorFeatureTransformer

# Apply feature transformation
feature_transformer = StartupInvestorFeatureTransformer(current_year=2026)
transformed_df = feature_transformer.transform(cleaned_df)

print(f"✅ Feature engineering completed")
print(f"📊 Original features: {cleaned_df.shape[1]}")
print(f"📈 Engineered features: {transformed_df.shape[1]}")

# Show new features
new_features = set(transformed_df.columns) - set(cleaned_df.columns)
print(f"\n🆕 New engineered features: {new_features}")

## Model Training

The system trains multiple models and selects the best performer:

### Models:
1. **Baseline Ridge**: Linear regression with L2 regularization
2. **Random Forest**: Ensemble of decision trees
3. **XGBoost**: Gradient boosting framework

### Training Process:
- Target transformation: log1p for better model performance
- Train/test split: 80/20 ratio
- MLflow tracking for experiment management
- Automatic model selection based on R² score

In [ ]:
from src.models.train import train_and_evaluate_all_models

# Train all models
print("🚀 Starting model training...")
results = train_and_evaluate_all_models()

print(f"\n🏆 Champion Model: {results['best_model_name']}")
print(f"📈 Best R² Score: {results['best_r2']:.4f}")
print(f"💾 Model exported to: {results['model_export_path']}")

# Show all model results
print("\n📊 All Model Results:")
for model_name, metrics in results['results_summary'].items():
    print(f"{model_name}: R²={metrics['R2']:.4f}, MAE=₹{metrics['MAE']:,.2f}")

## Model Evaluation

### Evaluation Metrics:
- **MAE**: Mean Absolute Error (₹)
- **RMSE**: Root Mean Square Error (₹)
- **R²**: Coefficient of determination
- **Within 20%**: Percentage of predictions within 20% of actual
- **Within 30%**: Percentage of predictions within 30% of actual

In [ ]:
# Load the trained model
model = joblib.load("artifacts/model.joblib")
print("✅ Model loaded successfully")

# Show model pipeline steps
print("\n🔧 Model Pipeline Steps:")
for step_name, step_obj in model.named_steps:
    print(f"- {step_name}: {type(step_obj).__name__}")

## Deployment

### Architecture:
```
Client (Streamlit) → API (FastAPI) → Model (sklearn Pipeline)
```

### Components:
- **FastAPI**: REST API for model serving
- **Streamlit**: Interactive web interface
- **Docker**: Containerized deployment
- **MLflow**: Model registry and tracking

### API Endpoints:
- `GET /health`: Health check
- `POST /predict`: Funding prediction
- `POST /explain`: Feature attribution
- `GET /metrics`: Service metrics

In [ ]:
# Example API prediction
import requests

# Sample prediction payload
sample_payload = {
    "investor_name": "Kalaari Capital",
    "investor_type": "VC",
    "headquarters_city": "Bengaluru",
    "headquarters_state": "Karnataka",
    "founded_year": 2015,
    "investment_stage": "Series A",
    "preferred_sector": "FinTech",
    "portfolio_companies": 60,
    "successful_exits": 15,
    "active_fund": 1,
    "ai_focus": 1,
    "fintech_focus": 1,
    "healthtech_focus": 0,
    "agritech_focus": 0
}

print("📤 Sample prediction payload prepared")
print(f"💰 Expected to predict funding for {sample_payload['investor_name']} in {sample_payload['investment_stage']} stage")

## Usage Examples

### Local Prediction:
Use the loaded model for local predictions without API.

In [ ]:
# Local prediction using the loaded model
input_df = pd.DataFrame([sample_payload])

# Make prediction
pred_log = model.predict(input_df)[0]
pred_inr = float(np.expm1(pred_log))

# Format result
def format_inr(amount):
    if amount >= 1_00_00_000:
        return f"₹{amount / 1_00_00_000:.2f} Crore"
    elif amount >= 1_00_000:
        return f"₹{amount / 1_00_000:.2f} Lakh"
    else:
        return f"₹{amount:.2f}"

print(f"🎯 Local Prediction Result:")
print(f"💰 Expected Funding: {format_inr(pred_inr)}")
print(f"📊 Raw Value: ₹{pred_inr:,.2f}")
print(f"🔢 Log-transformed: {pred_log:.4f}")

### Feature Importance Analysis:
Understand which features drive the predictions.

In [ ]:
from src.evaluation.explainability import get_global_feature_importance

# Get feature importance
feature_importance = get_global_feature_importance(model)

# Display top features
print("📊 Top 10 Most Important Features:")
for i, feat in enumerate(feature_importance[:10], 1):
    print(f"{i}. {feat['feature']}: {feat['importance']:.4f}")

## Conclusion

This notebook has covered:
1. ✅ Data pipeline and cleaning
2. ✅ Feature engineering process
3. ✅ Model training and evaluation
4. ✅ Deployment architecture
5. ✅ Usage examples

### Next Steps:
- Deploy the API and Streamlit app
- Monitor model performance in production
- Retrain model with new data periodically
- Explore additional features and model improvements

### Resources:
- **API Documentation**: Run the FastAPI server and visit `/docs`
- **Streamlit UI**: Run `streamlit run streamlit/app.py`
- **MLflow Tracking**: Run `mlflow ui` to view experiments